## Import Dataset

In [2]:
import pandas as pd

dataset = pd.read_csv('michelin_my_maps.csv')
print(dataset.head())


                           Name  \
0  8 1/2 Otto e Mezzo - Bombana   
1                          ABaC   
2       AM par Alexandre Mazzia   
3                       Addison   
4                       Akelaŕe   

                                             Address  \
0  Shop 202, 2F, Alexandra House, 18 Chater Road,...   
1    Avenida del Tibidabo 1, Barcelona, 08022, Spain   
2     9 rue François-Rocca, Marseille, 13008, France   
3  Fairmont Grand Del Mar, 5200 Grand Del Mar Way...   
4  Paseo del Padre Orcolaga 56, Donostia / San Se...   

                          Location Price       Cuisine   Longitude   Latitude  \
0         Hong Kong, Hong Kong SAR  $$$$       Italian  114.158670  22.281464   
1                 Barcelona, Spain  $$$$      Creative    2.136511  41.410450   
2                Marseille, France  $$$$      Creative    5.386233  43.270110   
3               San Diego, CA, USA  $$$$  Contemporary -117.198891  32.941297   
4  Donostia / San Sebastián, Spain  $$$$      C

## Clean Location Data
USA only, separate city, state, country into separate columns

In [3]:
parts = dataset['Location'].str.split(',').apply(lambda x: [p.strip() for p in x])

dataset['City'] = parts.str[0]
dataset['Country'] = parts.str[-1]

# State only for US rows with the "City, ST, USA" pattern
is_us = (dataset['Country'] == 'USA') & (parts.str.len() == 3)
dataset['State'] = parts.str[1].where(is_us)

print(dataset[['Location', 'City', 'State', 'Country']].head())

                          Location                      City State  \
0         Hong Kong, Hong Kong SAR                 Hong Kong   NaN   
1                 Barcelona, Spain                 Barcelona   NaN   
2                Marseille, France                 Marseille   NaN   
3               San Diego, CA, USA                 San Diego    CA   
4  Donostia / San Sebastián, Spain  Donostia / San Sebastián   NaN   

         Country  
0  Hong Kong SAR  
1          Spain  
2         France  
3            USA  
4          Spain  


In [4]:
usa = dataset[dataset['Country'] == 'USA'].copy()

print(usa.shape)
print(usa[['Name', 'City', 'State', 'Country']].head())

(1847, 17)
                   Name           City State Country
3               Addison      San Diego    CA     USA
13        Atelier Crenn  San Francisco    CA     USA
19                 Benu  San Francisco    CA     USA
22          Californios  San Francisco    CA     USA
37  Eleven Madison Park       New York    NY     USA


## Clean Star Rating Data
Convert award column to star count column (numerical) and BibGourmand Column (categorical - 0 = no, yes = 1)

In [5]:
star_map = {'3 Stars': 3, '2 Stars': 2, '1 Star': 1}

usa['Star Count'] = usa['Award'].map(star_map).fillna(0).astype(int)
usa['BibGourmand'] = (usa['Award'] == 'Bib Gourmand').astype(int)

print(usa[['Name', 'Award', 'Star Count', 'BibGourmand']].head())

                   Name    Award  Star Count  BibGourmand
3               Addison  3 Stars           3            0
13        Atelier Crenn  3 Stars           3            0
19                 Benu  3 Stars           3            0
22          Californios  3 Stars           3            0
37  Eleven Madison Park  3 Stars           3            0


In [6]:
usa = usa.drop(columns='Award')
print(usa[['Name', 'Star Count', 'BibGourmand']].head())

                   Name  Star Count  BibGourmand
3               Addison           3            0
13        Atelier Crenn           3            0
19                 Benu           3            0
22          Californios           3            0
37  Eleven Madison Park           3            0


## Clean Price Tier Data
Convert the `Price` column from `$` strings to a numeric tier (1–4).

| Symbol | Tier | Meaning | Cost per person |
|---|---|---|---|
| \$ | 1 | On a budget | < \$25 |
| \$\$ | 2 | A moderate spend | \$25–\$50 |
| \$\$\$ | 3 | Special occasion | \$50–\$75 |
| \$\$\$\$ | 4 | Spare no expense | > \$75 |

AI Disclosure: Used Claude to format this comment.

In [7]:
price_map = {'$': 1, '$$': 2, '$$$': 3, '$$$$': 4}

usa['Price Tier'] = usa['Price'].map(price_map).fillna(0).astype(int)

print(usa[['Name', 'Price', 'Price Tier',]].head())

                   Name Price  Price Tier
3               Addison  $$$$           4
13        Atelier Crenn  $$$$           4
19                 Benu  $$$$           4
22          Californios  $$$$           4
37  Eleven Madison Park  $$$$           4
